# LC GP publication plots (optional)

Run after `python 3_LCfit_KN_log.py`. Cells below produce multi-band figures under `Outputs/<SN>/plots-for-flash/`.


In [ ]:
import os
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime

rt = pconf.bootstrap_runtime(photometry_stage="extrapolated")
COCO_PATH = rt.coco_path
SNNAME = rt.snname
OUTPUT_DIR = rt.output_dir
DATALC_PATH = rt.datalc_path
color_dict, mark_dict = rt.color_dict, rt.mark_dict
SN = type('SN', (), {'snname': SNNAME, 'avail_filters': [], 'fitted_phot': {}, 'clipped_phot': None})()
# Load fitted products from step 3 script output; publication cells below expect SN-like object.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


### Final GP light-curve figure (publication block)

Run **`SN.LCfit_withGP()`** (or ensure **`SN.fitted_phot`** exists) before this cell. Edit the knobs in the next cell—filters, legend text, axis limits, figure size, and optional **`SAVE_FIG`**.

In [ ]:
# --- knobs: GP multi-band plot (log10 phase vs log10 flux) — same style as SN.save_plot_GPfit ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib

# Bands to plot: None = all SN.avail_filters (survey order); or pass an ordered list, e.g. ["Swope_g", "Swope_r", "Swope_i"]
FILTERS = None

# Legend strings per internal band key; bands not listed use LEGEND_LABEL_FUNC(...)
LEGEND_MAP = {
    # "Swope_g": r"$g$",
    # "DECam_r": r"DECam $r$",
}


def LEGEND_LABEL_FUNC(band_key: str) -> str:
    """Same default naming as save_plot_GPfit (Swift bands get a suffix)."""
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)


def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))


FIGSIZE = (14, 6)
DPI = 500

# Limits: None = auto (see USE_GLOBAL_* below)
XLIM = None  # e.g. (-3.0, 1.5)
YLIM = None  # e.g. (-19.0, -14.0)

# Match save_plot_GPfit: x/y ranges use full clipped table ± padding when True;
# when False, tighten to the bands actually drawn (better for a FILTER subset).
USE_GLOBAL_X_LIMITS = True
USE_GLOBAL_Y_LIMITS = True

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5
# Match save_plot_GPfit: SUDO points are plotted but usually omitted from the legend.
SHOW_SUDO_IN_LEGEND = False

LEGEND_KW = dict(
    fontsize=14,
    ncol=2,
    loc="best",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  # e.g. "Bands"

TITLE = None  # None → automatic from SN.snname

SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP.png'  # e.g. OUTPUT_DIR + SN.snname + "/figures/GP_LC_final.pdf"

# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)

plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

max_flux_tracker = [-np.inf]
log_phase_chunks = []
log_flux_chunks = []

for i, f in enumerate(bands):
    if f not in SN.fitted_phot:
        print(f"[skip] {f!r} not in SN.fitted_phot")
        continue

    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)

    mk = mark_dict.get(f, "o")
    lab = legend_label_for_band(f)

    _eb = ax.errorbar(
        log_phase[~sudo_mask],
        log_flux[~sudo_mask],
        yerr=err_log_flux[~sudo_mask],
        fmt=mk,
        mfc=c,
        ms=MARKER_MS,
        color=c,
        linestyle="None",
        label=lab,
    )
    data_ln = _eb[0]
    ax.errorbar(
        log_phase[sudo_mask],
        log_flux[sudo_mask],
        yerr=err_log_flux[sudo_mask],
        fmt=mk,
        mfc="white",
        mec=c,
        ms=MARKER_MS,
        mew=0.5,
        color=c,
        linestyle="None",
        label=lab + " (SUDO)" if np.any(sudo_mask) else "_nolegend_",
    )

    ax.plot(new_log_phase, mu, color=data_ln.get_color())
    ax.fill_between(
        new_log_phase,
        (mu + std),
        (mu - std),
        color=data_ln.get_color(),
        alpha=GP_FILL_ALPHA,
    )

    max_flux_tracker.append(float(np.nanmax(log_flux)))
    log_phase_chunks.append(log_phase)
    log_flux_chunks.append(log_flux)

max_flux_plot = float(np.max(max_flux_tracker))

if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        if YLIM is not None:
            y0, y1 = YLIM
        elif USE_GLOBAL_Y_LIMITS:
            y0 = float(np.min(SN.clipped_phot["Log_Flux"])) - 1.0
            y1 = max_flux_plot + 1.0
        else:
            flux_cat = np.concatenate(log_flux_chunks) if log_flux_chunks else SN.clipped_phot["Log_Flux"]
            y0 = float(np.nanmin(flux_cat)) - 1.0
            y1 = float(np.nanmax(flux_cat)) + 1.0
        ax.vlines(spec_log_phases, y0, y1, **SPECTRA_KW)

if XLIM is not None:
    ax.set_xlim(*XLIM)
elif USE_GLOBAL_X_LIMITS:
    try:
        ax.set_xlim(
            float(np.min(SN.clipped_phot["Log_Phase"])) - 0.2,
            float(np.max(SN.clipped_phot["Log_Phase"])) + 0.2,
        )
    except Exception:
        pass
else:
    if log_phase_chunks:
        pc = np.concatenate(log_phase_chunks)
        ax.set_xlim(float(np.nanmin(pc)) - 0.2, float(np.nanmax(pc)) + 0.2)

if YLIM is not None:
    ax.set_ylim(*YLIM)
elif USE_GLOBAL_Y_LIMITS:
    ax.set_ylim(float(np.min(SN.clipped_phot["Log_Flux"])) - 1.0, max_flux_plot + 1.0)
else:
    if log_flux_chunks:
        fc = np.concatenate(log_flux_chunks)
        ax.set_ylim(float(np.nanmin(fc)) - 1.0, float(np.nanmax(fc)) + 1.0)

ax.set_xlabel(r"Log Phase relative to explosion (days)", fontsize=16)
ax.set_ylabel(
    r"Log Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)",
    fontsize=16,
)
#ax.minorticks_on()
ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

tit = TITLE if TITLE is not None else (
    SN.snname + " light curve fitting using Gaussian Processes (Log-Space)"
)
#ax.set_title(tit, fontsize=15)

# #leg = ax.legend(**LEGEND_KW)
# if LEGEND_TITLE is not None:
#     leg.set_title(LEGEND_TITLE)

# --- Custom Legend Setup ---
# Create proxy artists for the overarching data types. 
# Using 'gray' or 'black' keeps it color-neutral to represent all bands.
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
           
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', 
           markeredgecolor='black', markeredgewidth=1, 
           markersize=MARKER_MS+2, label='Added Points'),
           
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

# Pass the custom handles directly to the legend
leg = ax.legend(handles=custom_handles, **LEGEND_KW)

if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

#matplotlib.ticker.LogLocator(base=10, subs='all')

plt.tight_layout()


if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=500, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (log10 phase vs log10 flux) — same style as SN.save_plot_GPfit ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

# Bands to plot: None = all SN.avail_filters (survey order); or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    """Same default naming as save_plot_GPfit (Swift bands get a suffix)."""
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

FIGSIZE = (10, 5) # Slightly taller to accommodate the vertical offsets
DPI = 500

# ==========================================
# NEW WATERFALL & SPACE CONTROLS
# ==========================================
OFFSET_PER_BAND = 0      # Offset applied to each successive band in log space (dex)
PLOT_Y_LINEAR = True      # True = plot linear Flux, False = plot Log Flux
PLOT_X_LINEAR = False       # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False       # True = labels the band name at the end of its GP curve
# ==========================================

XLIM = -3, 1.6
# YLIM = -20, 20 
#XLIM = 0, 20
YLIM = 0, 1.5e-15

USE_GLOBAL_X_LIMITS = False # Changed to False to let limits adapt to the new offsets
USE_GLOBAL_Y_LIMITS = False 

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5
SHOW_SUDO_IN_LEGEND = False

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, # Switched to 1 column for the 3-item custom legend
    loc="best",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

#SAVE_FIG = None
SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_linearflux_logtime.png'

# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)

plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

log_phase_chunks = []
log_flux_chunks = []

for i, f in enumerate(bands):
    if f not in SN.fitted_phot:
        print(f"[skip] {f!r} not in SN.fitted_phot")
        continue

    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)
    
    good = ~sudo_mask
    sudo = sudo_mask

    lab = legend_label_for_band(f)
    mk = mark_dict.get(f, "o")

    # Calculate current offset (applied iteratively per band)
    current_offset = i * OFFSET_PER_BAND

    # ---------------------------------------------------------
    # Space Transformations (Linear vs Log) + Offsets
    # ---------------------------------------------------------
    if PLOT_X_LINEAR:
        x_plot = 10**log_phase
        x_gp_plot = 10**new_log_phase
    else:
        x_plot = log_phase
        x_gp_plot = new_log_phase

    if PLOT_Y_LINEAR:
        # Convert log flux to linear flux. The offset acts as a scalar: 10**(log_flux + offset)
        y_plot = 10**(log_flux + current_offset)
        
        # Calculate asymmetric errors for linear space
        y_err_lower = y_plot - 10**(log_flux - err_log_flux + current_offset)
        y_err_upper = 10**(log_flux + err_log_flux + current_offset) - y_plot
        y_err_plot = np.array([y_err_lower, y_err_upper])
        
        # Split errors based on masks (shape handling for 2D error arrays)
        y_err_good = y_err_plot[:, good] if np.any(good) else y_err_plot
        y_err_sudo = y_err_plot[:, sudo] if np.any(sudo) else y_err_plot
        
        # Transform GP fit
        y_gp_mu_plot = 10**(mu + current_offset)
        y_gp_upper = 10**(mu + std + current_offset)
        y_gp_lower = 10**(mu - std + current_offset)
        
    else:
        # Standard log space plotting with additive offsets
        y_plot = log_flux + current_offset
        y_err_plot = err_log_flux
        
        y_err_good = y_err_plot[good]
        y_err_sudo = y_err_plot[sudo]
        
        y_gp_mu_plot = mu + current_offset
        y_gp_upper = mu + std + current_offset
        y_gp_lower = mu - std + current_offset
    # ---------------------------------------------------------

    # Track data for axis limits
    log_phase_chunks.append(x_plot)
    log_flux_chunks.append(y_plot)

    # Plot actual photometry
    if np.any(good):
        _eb = ax.errorbar(
            x_plot[good], y_plot[good], yerr=y_err_good,
            fmt=mk, mfc=c, ms=MARKER_MS, color=c,
            linestyle="None", label="_nolegend_"
        )
        data_color = _eb[0].get_color()
    else:
        data_color = c

    # Plot added SUDO points
    if np.any(sudo):
        ax.errorbar(
            x_plot[sudo], y_plot[sudo], yerr=y_err_sudo,
            fmt=mk, mfc="white", mec=data_color, ms=MARKER_MS, mew=0.5,
            color=data_color, linestyle="None", label="_nolegend_"
        )

    # Plot GP fit
    ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
    ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

    # Annotate band names right next to the lines
    if ANNOTATE_BANDS and len(x_gp_plot) > 0:
        ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.5), y_gp_mu_plot[-1], 
                f"{lab}", color=data_color, fontsize=12, va='center', ha='left')


#Automatically set axes based on transformed chunk data
# if log_phase_chunks and log_flux_chunks:
#     pc = np.concatenate(log_phase_chunks)
#     fc = np.concatenate(log_flux_chunks)
#     if XLIM is None and not USE_GLOBAL_X_LIMITS:
#         ax.set_xlim(float(np.nanmin(pc)) - (0.2 if not PLOT_X_LINEAR else 1.0), 
#                     float(np.nanmax(pc)) + (0.4 if not PLOT_X_LINEAR else 3.0)) # Extra padding for annotations
#     if YLIM is None and not USE_GLOBAL_Y_LIMITS:
#         pad = 1.0 if not PLOT_Y_LINEAR else (float(np.nanmax(fc)) * 0.1)
#         ax.set_ylim(float(np.nanmin(fc)) - pad, float(np.nanmax(fc)) + pad)
    ax.set_xlim(XLIM)
    ax.set_ylim(YLIM)

# Spectra handling (omitted automatic scaling adjustments here for brevity if it's False)
if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        y0, y1 = ax.get_ylim()
        x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
        ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

# Set conditional Axis Labels
x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion (days)"
ax.set_xlabel(x_label, fontsize=16)

if PLOT_Y_LINEAR:
    y_label = r"Scaled Flux" if OFFSET_PER_BAND != 0 else r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
ax.set_ylabel(y_label, fontsize=16)


ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

if TITLE is not None:
    ax.set_title(TITLE, fontsize=15)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', 
           markeredgecolor='black', markeredgewidth=1, 
           markersize=MARKER_MS+2, label='Added Points'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = ax.legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

plt.tight_layout()

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (linear phase vs linear flux) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()
# Bands to plot: None = all SN.avail_filters (survey order); or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    """Same default naming as save_plot_GPfit (Swift bands get a suffix)."""
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

FIGSIZE = (10, 5) 
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
OFFSET_PER_BAND = 0          # Offset applied to each successive band in log space (dex)
PLOT_Y_LINEAR = True         # True = plot linear Flux, False = plot Log Flux
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False       # True = labels the band name at the end of its GP curve

# NEW CONTROL: Truncate the GP fit?
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
#XLIM = (0, 20)      # e.g. None, or (-3, 1.6)
#YLIM = (0, 1.5e-15) # e.g. None, or (-20, 20)
XLIM = 0, 5
YLIM = None

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="best",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_linear.png'

# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)

plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

for i, f in enumerate(bands):
    if f not in SN.fitted_phot:
        print(f"[skip] {f!r} not in SN.fitted_phot")
        continue

    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)
    
    # We only care about the actual photometry now
    good = ~sudo_mask

    lab = legend_label_for_band(f)
    mk = mark_dict.get(f, "o")

    current_offset = i * OFFSET_PER_BAND

    # ---------------------------------------------------------
    # Space Transformations (Linear vs Log) + Offsets
    # ---------------------------------------------------------
    if PLOT_X_LINEAR:
        x_plot = 10**log_phase
        x_gp_plot = 10**new_log_phase
    else:
        x_plot = log_phase
        x_gp_plot = new_log_phase

    if PLOT_Y_LINEAR:
        y_plot = 10**(log_flux + current_offset)
        
        y_err_lower = y_plot - 10**(log_flux - err_log_flux + current_offset)
        y_err_upper = 10**(log_flux + err_log_flux + current_offset) - y_plot
        y_err_plot = np.array([y_err_lower, y_err_upper])
        
        y_err_good = y_err_plot[:, good] if np.any(good) else y_err_plot
        
        y_gp_mu_plot = 10**(mu + current_offset)
        y_gp_upper = 10**(mu + std + current_offset)
        y_gp_lower = 10**(mu - std + current_offset)
        
    else:
        y_plot = log_flux + current_offset
        y_err_plot = err_log_flux
        
        y_err_good = y_err_plot[good]
        
        y_gp_mu_plot = mu + current_offset
        y_gp_upper = mu + std + current_offset
        y_gp_lower = mu - std + current_offset
    # ---------------------------------------------------------

    # If truncating GP fit, slice the arrays using the min and max of actual data
    if TRUNCATE_GP_TO_DATA and np.any(good):
        min_x = np.min(x_plot[good])
        max_x = np.max(x_plot[good])
        
        valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
        
        x_gp_plot = x_gp_plot[valid_gp_mask]
        y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
        y_gp_upper = y_gp_upper[valid_gp_mask]
        y_gp_lower = y_gp_lower[valid_gp_mask]

    # Plot actual photometry
    if np.any(good):
        _eb = ax.errorbar(
            x_plot[good], y_plot[good], yerr=y_err_good,
            fmt=mk, mfc=c, ms=MARKER_MS, color=c,
            linestyle="None", label="_nolegend_"
        )
        data_color = _eb[0].get_color()
    else:
        data_color = c

    # Plot GP fit (SUDO points removed)
    if len(x_gp_plot) > 0:
        ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
        ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

    # Annotate band names right next to the lines
    if ANNOTATE_BANDS and len(x_gp_plot) > 0:
        ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.5), y_gp_mu_plot[-1], 
                f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

# Handle manual or automatic limits
if XLIM is not None:
    ax.set_xlim(XLIM)
if YLIM is not None:
    ax.set_ylim(YLIM)

# Spectra handling
if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        y0, y1 = ax.get_ylim()
        x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
        ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

# Set conditional Axis Labels
x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion ($\log_{10}(\rm{days})$)"
ax.set_xlabel(x_label, fontsize=16)

if PLOT_Y_LINEAR:
    y_label = r"Scaled Flux" if OFFSET_PER_BAND != 0 else r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
ax.set_ylabel(y_label, fontsize=16)

ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

if TITLE is not None:
    ax.set_title(TITLE, fontsize=15)

# --- Custom Legend Setup ---
# Removed the SUDO points handle
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = ax.legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

plt.tight_layout()

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (linear phase vs linear flux, dynamically sorted) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

# Bands to plot: None = all SN.avail_filters; or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

FIGSIZE = (10, 8) 
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
OFFSET_PER_BAND = 0.05    # Tweak this to change the vertical spacing (e.g., -0.3 is tighter, -0.6 is wider)
PLOT_Y_LINEAR = True         # True = plot linear Flux, False = plot Log Flux
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False        # True = labels the band name at the end of its GP curve

# DYNAMIC SORTING CONTROLS
SORT_BY_EARLY_PEAK = True    # True = sorts bands so highest early peaks are plotted at the top
EARLY_TIME_CUTOFF = 2.0      # The phase (in days) to search for the maximum flux for sorting

# TRUNCATION
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
XLIM = (0, 5)      
YLIM = (0, 6e-14) 

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="upper right",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

#SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_linear_sorted.png'
SAVE_FIG = None
# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

# 1. Gather all available bands
initial_bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)
valid_bands = [b for b in initial_bands if b in SN.fitted_phot]

# 2. Pre-calculate early-time peaks for dynamic sorting
if SORT_BY_EARLY_PEAK:
    early_peaks = {}
    for f in valid_bands:
        log_phase, log_flux, _, _ = SN.fitted_phot[f]["clipped_extended_data"]
        
        # Convert to linear to evaluate the early time window
        phase_lin = 10**np.asarray(log_phase, dtype=float)
        flux_lin = 10**np.asarray(log_flux, dtype=float)
        
        # Find maximum flux within the first X days
        early_mask = phase_lin <= EARLY_TIME_CUTOFF
        if np.any(early_mask):
            peak = np.nanmax(flux_lin[early_mask])
        else:
            # Fallback if no data exists in the first 2 days
            peak = np.nanmax(flux_lin) if len(flux_lin) > 0 else -np.inf
            
        early_peaks[f] = peak

    # Sort valid bands based on their early peak (highest peak goes first, index 0, offset 0)
    plot_bands = sorted(valid_bands, key=lambda b: early_peaks[b], reverse=True)
else:
    plot_bands = valid_bands


plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

for i, f in enumerate(plot_bands):
    
    # Grab the original color from the color dict (or default if missing)
    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)
    
    # Only dealing with actual photometry
    good = ~sudo_mask

    lab = legend_label_for_band(f)
    mk = mark_dict.get(f, "o")

    # Offset applied iteratively: 0 for the brightest band, -0.4 for the next, -0.8 for the next, etc.
    current_offset = i * OFFSET_PER_BAND

    # ---------------------------------------------------------
    # Space Transformations + Offsets
    # ---------------------------------------------------------
    if PLOT_X_LINEAR:
        x_plot = 10**log_phase
        x_gp_plot = 10**new_log_phase
    else:
        x_plot = log_phase
        x_gp_plot = new_log_phase

    if PLOT_Y_LINEAR:
        y_plot = 10**(log_flux + current_offset)
        
        y_err_lower = y_plot - 10**(log_flux - err_log_flux + current_offset)
        y_err_upper = 10**(log_flux + err_log_flux + current_offset) - y_plot
        y_err_plot = np.array([y_err_lower, y_err_upper])
        
        y_err_good = y_err_plot[:, good] if np.any(good) else y_err_plot
        
        y_gp_mu_plot = 10**(mu + current_offset)
        y_gp_upper = 10**(mu + std + current_offset)
        y_gp_lower = 10**(mu - std + current_offset)
        
    else:
        y_plot = log_flux + current_offset
        y_err_plot = err_log_flux
        y_err_good = y_err_plot[good]
        
        y_gp_mu_plot = mu + current_offset
        y_gp_upper = mu + std + current_offset
        y_gp_lower = mu - std + current_offset
    # ---------------------------------------------------------

    # Slice GP fit down to only span the bounds of the real data
    if TRUNCATE_GP_TO_DATA and np.any(good):
        min_x = np.min(x_plot[good])
        max_x = np.max(x_plot[good])
        
        valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
        
        x_gp_plot = x_gp_plot[valid_gp_mask]
        y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
        y_gp_upper = y_gp_upper[valid_gp_mask]
        y_gp_lower = y_gp_lower[valid_gp_mask]

    # Plot actual photometry
    if np.any(good):
        _eb = ax.errorbar(
            x_plot[good], y_plot[good], yerr=y_err_good,
            fmt=mk, mfc=c, ms=MARKER_MS, color=c,
            linestyle="None", label="_nolegend_"
        )
        data_color = _eb[0].get_color()
    else:
        data_color = c

    # Plot GP fit
    if len(x_gp_plot) > 0:
        ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
        ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

    # Annotate band names (highly recommended for waterfall plots to avoid legend hunting)
    if ANNOTATE_BANDS and len(x_gp_plot) > 0:
        ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.3), y_gp_mu_plot[-1], 
                f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

# Handle manual limits
if XLIM is not None:
    ax.set_xlim(XLIM)
if YLIM is not None:
    ax.set_ylim(YLIM)

# Spectra handling
if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        y0, y1 = ax.get_ylim()
        x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
        ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

# Set Axis Labels
x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion ($\log_{10}(\rm{days})$)"
ax.set_xlabel(x_label, fontsize=16)

if PLOT_Y_LINEAR:
    y_label = r"Scaled Flux" if OFFSET_PER_BAND != 0 else r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
ax.set_ylabel(y_label, fontsize=16)

ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

if TITLE is not None:
    ax.set_title(TITLE, fontsize=15)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = ax.legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

plt.tight_layout()

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (linear phase vs linear flux, sorted blue to red) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib

# Bands to plot: None = all SN.avail_filters; or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

FIGSIZE = (14, 10) # Taller to accommodate stacked linear bands
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
# For linear plots, this is now an ADDITIVE offset (e.g., 2e-17 erg/s/cm2/A)
OFFSET_PER_BAND = 0     

PLOT_Y_LINEAR = True         # True = plot linear Flux, False = plot Log Flux
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False        # True = labels the band name at the end of its GP curve

# DYNAMIC SORTING CONTROLS
SORT_BLUE_TO_RED = True      # True = sorts bands from shortest to longest wavelength

# TRUNCATION
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
XLIM = 0, 5    
YLIM = None # Let it auto-scale based on the new additive offsets

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="upper right",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

#SAVE_FIG = None # e.g. '/path/to/save.png'
SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_linear_xlim.png'


# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

# 1. Gather all available bands
initial_bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)
valid_bands = [b for b in initial_bands if b in SN.fitted_phot]

# 2. Sort bands from bluest to reddest
if SORT_BLUE_TO_RED:
    # Approximate central wavelengths in Angstroms for sorting
    WAVE_DICT = {
        'w2': 1928, 'm2': 2246, 'w1': 2600, 
        'u': 3543, 'b': 4450, 'g': 4770, 'v': 5510, 
        'r': 6231, 'i': 7625, 'z': 9134, 'y': 10040, 
        'j': 12200, 'h': 16300, 'k': 21900, 'ks': 21900
    }
    
    def get_wave(band_name):
        # Extract the base filter letter (e.g., 'Swope_r' -> 'r')
        b_lower = band_name.lower().split('_')[-1]
        # Check against swift explicitly to catch swift_v, etc.
        if 'swift' in band_name.lower():
            b_lower = band_name.lower().split('_')[1]
            
        for key, wave in sorted(WAVE_DICT.items(), key=lambda item: len(item[0]), reverse=True):
            if key in b_lower:
                return wave
        return 99999 # Push unrecognized bands to the bottom

    plot_bands = sorted(valid_bands, key=get_wave)
else:
    plot_bands = valid_bands


plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

for i, f in enumerate(plot_bands):
    
    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)
    
    good = ~sudo_mask

    lab = legend_label_for_band(f)
    mk = mark_dict.get(f, "o")

    # The offset to apply for this specific band
    current_offset = i * OFFSET_PER_BAND

    # ---------------------------------------------------------
    # Space Transformations + ADDITIVE Offsets
    # ---------------------------------------------------------
    if PLOT_X_LINEAR:
        x_plot = 10**log_phase
        x_gp_plot = 10**new_log_phase
    else:
        x_plot = log_phase
        x_gp_plot = new_log_phase

    if PLOT_Y_LINEAR:
        # 1. Convert to linear flux FIRST
        linear_flux = 10**log_flux
        
        # 2. Add the constant linear offset so the baselines visually separate
        y_plot = linear_flux + current_offset
        
        # Errors in linear space (additive offset doesn't change the error bar size)
        y_err_lower = linear_flux - 10**(log_flux - err_log_flux)
        y_err_upper = 10**(log_flux + err_log_flux) - linear_flux
        y_err_plot = np.array([y_err_lower, y_err_upper])
        y_err_good = y_err_plot[:, good] if np.any(good) else y_err_plot
        
        # Transform GP fit and apply additive offset
        y_gp_mu_plot = 10**(mu) + current_offset
        y_gp_upper = 10**(mu + std) + current_offset
        y_gp_lower = 10**(mu - std) + current_offset
        
    else:
        # Standard log space additive offsets
        y_plot = log_flux + current_offset
        y_err_plot = err_log_flux
        y_err_good = y_err_plot[good]
        
        y_gp_mu_plot = mu + current_offset
        y_gp_upper = mu + std + current_offset
        y_gp_lower = mu - std + current_offset
    # ---------------------------------------------------------

    if TRUNCATE_GP_TO_DATA and np.any(good):
        min_x = np.min(x_plot[good])
        max_x = np.max(x_plot[good])
        
        valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
        
        x_gp_plot = x_gp_plot[valid_gp_mask]
        y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
        y_gp_upper = y_gp_upper[valid_gp_mask]
        y_gp_lower = y_gp_lower[valid_gp_mask]

    # Plot actual photometry
    if np.any(good):
        _eb = ax.errorbar(
            x_plot[good], y_plot[good], yerr=y_err_good,
            fmt=mk, mfc=c, ms=MARKER_MS, color=c,
            linestyle="None", label="_nolegend_"
        )
        data_color = _eb[0].get_color()
    else:
        data_color = c

    # Plot GP fit
    if len(x_gp_plot) > 0:
        ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
        ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

    # Annotate band names
    if ANNOTATE_BANDS and len(x_gp_plot) > 0:
        ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.15), y_gp_mu_plot[-1], 
                f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

if XLIM is not None:
    ax.set_xlim(XLIM)
if YLIM is not None:
    ax.set_ylim(YLIM)

if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        y0, y1 = ax.get_ylim()
        x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
        ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion (days))"
ax.set_xlabel(x_label, fontsize=16)

if PLOT_Y_LINEAR:
    y_label = r"Scaled Flux + Offset" if OFFSET_PER_BAND != 0 else r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)"
ax.set_ylabel(y_label, fontsize=16)

ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

if TITLE is not None:
    ax.set_title(TITLE, fontsize=15)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = ax.legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

plt.tight_layout()

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (linear phase vs magnitude, sorted blue to red) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib

# Bands to plot: None = all SN.avail_filters; or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

FIGSIZE = (10, 30) # Taller to accommodate stacked bands
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
# Offset for magnitudes (e.g., 1.5 or 2.0 works well for staggering bands)
OFFSET_PER_BAND = 0.5     

PLOT_Y_MAGNITUDE = True      # True = plot Magnitude (Proper AB mag), False = plot Log Flux
INVERT_Y_AXIS = True         # True = Inverts the Y axis (standard for magnitudes so brighter is up)
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False       # True = labels the band name at the end of its GP curve

# DYNAMIC SORTING CONTROLS
SORT_BLUE_TO_RED = True      # True = sorts bands from shortest to longest wavelength

# TRUNCATION
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
XLIM = (0, 10)    
YLIM = None 

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="upper right",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

SAVE_FIG = None # e.g. '/path/to/save.png'
#SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_mag_xlim.png'

# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

# 1. Gather all available bands
initial_bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)
valid_bands = [b for b in initial_bands if b in SN.fitted_phot]

# 2. Approximate central wavelengths in Angstroms (Moved to global scope for magnitude math)
WAVE_DICT = {
    'w2': 1928, 'm2': 2246, 'w1': 2600, 
    'u': 3543, 'b': 4450, 'g': 4770, 'v': 5510, 
    'r': 6231, 'i': 7625, 'z': 9134, 'y': 10040, 
    'j': 12200, 'h': 16300, 'k': 21900, 'ks': 21900
}

def get_wave(band_name):
    b_lower = band_name.lower().split('_')[-1]
    if 'swift' in band_name.lower():
        b_lower = band_name.lower().split('_')[1]
        
    for key, wave in sorted(WAVE_DICT.items(), key=lambda item: len(item[0]), reverse=True):
        if key in b_lower:
            return wave
            
    print(f"Warning: No wavelength found for {band_name}. Defaulting to 6000A for math.")
    return 6000.0

# 3. Sort bands from bluest to reddest
if SORT_BLUE_TO_RED:
    plot_bands = sorted(valid_bands, key=get_wave)
else:
    plot_bands = valid_bands


plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

_prop_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
_default_colors = _prop_cycle if isinstance(_prop_cycle, list) else list(_prop_cycle)

for i, f in enumerate(plot_bands):
    
    c = color_dict.get(f)
    if c is None:
        c = _default_colors[i % len(_default_colors)]

    log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
    new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

    log_phase = np.asarray(log_phase, dtype=float)
    log_flux = np.asarray(log_flux, dtype=float)
    err_log_flux = np.asarray(err_log_flux, dtype=float)
    sudo_mask = np.asarray(sudo_mask, dtype=bool)
    
    good = ~sudo_mask

    lab = legend_label_for_band(f)
    mk = mark_dict.get(f, "o")

    # The offset to apply for this specific band
    current_offset = i * OFFSET_PER_BAND

    # ---------------------------------------------------------
    # Space Transformations + Proper AB Magnitude Conversion
    # ---------------------------------------------------------
    if PLOT_X_LINEAR:
        x_plot = 10**log_phase
        x_gp_plot = 10**new_log_phase
    else:
        x_plot = log_phase
        x_gp_plot = new_log_phase

    if PLOT_Y_MAGNITUDE:
        # Get effective wavelength and calculate the AB zero-point correction
        eff_wave = get_wave(f)
        zp_correction = -5.0 * np.log10(eff_wave) - 2.408
        
        # Convert F_lambda to true AB magnitude
        y_plot = -2.5 * log_flux + zp_correction + current_offset
        
        # Errors in magnitude space
        y_err_plot = 2.5 * err_log_flux
        y_err_good = y_err_plot[good]
        
        # Transform GP fit
        y_gp_mu_plot = -2.5 * mu + zp_correction + current_offset
        y_gp_lower = -2.5 * (mu + std) + zp_correction + current_offset
        y_gp_upper = -2.5 * (mu - std) + zp_correction + current_offset
        
    else:
        # Standard log space additive offsets
        y_plot = log_flux + current_offset
        y_err_plot = err_log_flux
        y_err_good = y_err_plot[good]
        
        y_gp_mu_plot = mu + current_offset
        y_gp_upper = mu + std + current_offset
        y_gp_lower = mu - std + current_offset
    # ---------------------------------------------------------

    if TRUNCATE_GP_TO_DATA and np.any(good):
        min_x = np.min(x_plot[good])
        max_x = np.max(x_plot[good])
        
        valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
        
        x_gp_plot = x_gp_plot[valid_gp_mask]
        y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
        y_gp_upper = y_gp_upper[valid_gp_mask]
        y_gp_lower = y_gp_lower[valid_gp_mask]

    # Plot actual photometry
    if np.any(good):
        _eb = ax.errorbar(
            x_plot[good], y_plot[good], yerr=y_err_good,
            fmt=mk, mfc=c, ms=MARKER_MS, color=c,
            linestyle="None", label="_nolegend_"
        )
        data_color = _eb[0].get_color()
    else:
        data_color = c

    # Plot GP fit
    if len(x_gp_plot) > 0:
        ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
        ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

    # Annotate band names
    if ANNOTATE_BANDS and len(x_gp_plot) > 0:
        ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.15), y_gp_mu_plot[-1], 
                f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

if XLIM is not None:
    ax.set_xlim(XLIM)
if YLIM is not None:
    ax.set_ylim(YLIM)

# Apply standard magnitude axis inversion
if INVERT_Y_AXIS and PLOT_Y_MAGNITUDE:
    ax.invert_yaxis()

if SHOW_SPECTRA_LINES:
    spec_log_phases = SN.get_spec_log_phase()
    if len(spec_log_phases) > 0:
        y0, y1 = ax.get_ylim()
        x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
        ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion (days)"
ax.set_xlabel(x_label, fontsize=16)

if PLOT_Y_MAGNITUDE:
    y_label = r"Apparent AB Magnitude + Offset" if OFFSET_PER_BAND != 0 else r"Apparent AB Magnitude"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
ax.set_ylabel(y_label, fontsize=16)

ax.tick_params(axis='both', which='major', labelsize=14)
ax.tick_params(axis='both', which='minor', labelsize=14)
ax.grid(True)

if TITLE is not None:
    ax.set_title(TITLE, fontsize=15)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = ax.legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

plt.tight_layout()

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (3-Panel 16:9, linear phase vs magnitude + Colorbar) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib

# Bands to plot: None = all SN.avail_filters; or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

# Exactly 16:9 aspect ratio for presentations
FIGSIZE = (16, 9) 
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
# Offset for magnitudes (e.g., 1.5 or 2.0 works well for staggering bands)
OFFSET_PER_BAND = 1.5     

PLOT_Y_MAGNITUDE = True      # True = plot Magnitude (Proper AB mag), False = plot Log Flux
INVERT_Y_AXIS = True         # True = Inverts the Y axis (standard for magnitudes so brighter is up)
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False       # True = labels the band name at the end of its GP curve

# DYNAMIC SORTING CONTROLS
SORT_BLUE_TO_RED = True      # True = sorts bands from shortest to longest wavelength

# TRUNCATION
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
XLIM = (0, 10)    
YLIM = (15, 55)

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="upper left",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

SAVE_FIG = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/LC_GP_panels_colorbar.png'
# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

# 1. Gather all available bands
initial_bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)
valid_bands = [b for b in initial_bands if b in SN.fitted_phot]

# 2. Approximate central wavelengths in Angstroms
WAVE_DICT = {
    'w2': 1928, 'm2': 2246, 'w1': 2600, 
    'u': 3543, 'b': 4450, 'g': 4770, 'v': 5510, 
    'r': 6231, 'i': 7625, 'z': 9134, 'y': 10040, 
    'j': 12200, 'h': 16300, 'k': 21900, 'ks': 21900
}

def get_wave(band_name):
    b_lower = band_name.lower().split('_')[-1]
    if 'swift' in band_name.lower():
        b_lower = band_name.lower().split('_')[1]
        
    for key, wave in sorted(WAVE_DICT.items(), key=lambda item: len(item[0]), reverse=True):
        if key in b_lower:
            return wave
            
    print(f"Warning: No wavelength found for {band_name}. Defaulting to 6000A for math.")
    return 6000.0

# 3. Sort bands from bluest to reddest
if SORT_BLUE_TO_RED:
    plot_bands = sorted(valid_bands, key=get_wave)
else:
    plot_bands = valid_bands

# 4. Split bands into 3 equal (or near-equal) chunks for the 3 panels
band_chunks = np.array_split(plot_bands, 3)

plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

# 5. Initialize 3-panel figure with shared y-axis
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, dpi=DPI, sharey=True)
norm = plt.Normalize(vmin=1500, vmax=22000)

for ax_idx, (ax, chunk) in enumerate(zip(axes, band_chunks)):
    
    for i, f in enumerate(chunk):
        
        # Force color mapping entirely by wavelength (bluest to reddest)
        wave = get_wave(f)
        c = cm.Spectral_r(norm(wave))

        log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
        new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

        log_phase = np.asarray(log_phase, dtype=float)
        log_flux = np.asarray(log_flux, dtype=float)
        err_log_flux = np.asarray(err_log_flux, dtype=float)
        sudo_mask = np.asarray(sudo_mask, dtype=bool)
        
        good = ~sudo_mask

        lab = legend_label_for_band(f)
        try:
            mk = mark_dict[f]
        except (NameError, KeyError):
            mk = "o"

        # Local offset (resets for each panel so they stack identically)
        current_offset = i * OFFSET_PER_BAND

        # ---------------------------------------------------------
        # Space Transformations + Proper AB Magnitude Conversion
        # ---------------------------------------------------------
        if PLOT_X_LINEAR:
            x_plot = 10**log_phase
            x_gp_plot = 10**new_log_phase
        else:
            x_plot = log_phase
            x_gp_plot = new_log_phase

        if PLOT_Y_MAGNITUDE:
            eff_wave = get_wave(f)
            zp_correction = -5.0 * np.log10(eff_wave) - 2.408
            
            y_plot = -2.5 * log_flux + zp_correction + current_offset
            y_err_plot = 2.5 * err_log_flux
            y_err_good = y_err_plot[good]
            
            y_gp_mu_plot = -2.5 * mu + zp_correction + current_offset
            y_gp_lower = -2.5 * (mu + std) + zp_correction + current_offset
            y_gp_upper = -2.5 * (mu - std) + zp_correction + current_offset
            
        else:
            y_plot = log_flux + current_offset
            y_err_plot = err_log_flux
            y_err_good = y_err_plot[good]
            
            y_gp_mu_plot = mu + current_offset
            y_gp_upper = mu + std + current_offset
            y_gp_lower = mu - std + current_offset
        # ---------------------------------------------------------

        if TRUNCATE_GP_TO_DATA and np.any(good):
            min_x = np.min(x_plot[good])
            max_x = np.max(x_plot[good])
            
            valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
            
            x_gp_plot = x_gp_plot[valid_gp_mask]
            y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
            y_gp_upper = y_gp_upper[valid_gp_mask]
            y_gp_lower = y_gp_lower[valid_gp_mask]

        # Plot actual photometry
        if np.any(good):
            _eb = ax.errorbar(
                x_plot[good], y_plot[good], yerr=y_err_good,
                fmt=mk, mfc=c, ms=MARKER_MS, color=c,
                linestyle="None", label="_nolegend_"
            )
            data_color = _eb[0].get_color()
        else:
            data_color = c

        # Plot GP fit
        if len(x_gp_plot) > 0:
            ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
            ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

        # Annotate band names
        if ANNOTATE_BANDS and len(x_gp_plot) > 0:
            ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.15), y_gp_mu_plot[-1], 
                    f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

    if XLIM is not None:
        ax.set_xlim(XLIM)
        
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis='both', which='minor', labelsize=14)
    ax.grid(True)
    
    if SHOW_SPECTRA_LINES:
        spec_log_phases = SN.get_spec_log_phase()
        if len(spec_log_phases) > 0:
            y0, y1 = ax.get_ylim()
            x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
            ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

# Handle Global Y-Axis Limits and Inversion on the first axis (it will propogate to all due to sharey=True)
if YLIM is not None:
    axes[0].set_ylim(YLIM)

if INVERT_Y_AXIS and PLOT_Y_MAGNITUDE:
    axes[0].invert_yaxis()

# Set Labels (X on middle plot, Y on left plot)
x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion (days)"
axes[1].set_xlabel(x_label, fontsize=18)

if PLOT_Y_MAGNITUDE:
    y_label = r"Apparent AB Magnitude + Offset" if OFFSET_PER_BAND != 0 else r"Apparent AB Magnitude"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
axes[0].set_ylabel(y_label, fontsize=18)

if TITLE is not None:
    fig.suptitle(TITLE, fontsize=18)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = axes[0].legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

# --- NEW: Explicit Layout and Colorbar Creation ---

sm = plt.cm.ScalarMappable(cmap=cm.Spectral_r, norm=norm)
sm.set_array([])

# 1. Apply tight_layout FIRST, explicitly reserving the right 8% of the figure for the colorbar
fig.tight_layout(rect=[0, 0, 0.92, 1], w_pad=0.0)

# 2. Get the physical coordinates of the right-most subplot
pos = axes[-1].get_position()

# 3. Create a dedicated axis for the colorbar: [left, bottom, width, height]
# We use the bottom (y0) and height from axes[-1] to guarantee perfect vertical alignment
cbar_ax = fig.add_axes([0.93, pos.y0, 0.015, pos.height])

# 4. Draw the colorbar into that dedicated axis
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(r'Effective Wavelength ($\mathrm{\AA}$)', fontsize=18)
cbar.ax.tick_params(labelsize=14)

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()


In [ ]:
# --- knobs: GP multi-band plot (3-Panel 16:9, linear phase vs magnitude + Colorbar) ---
# Requires SN.fitted_phot from SN.LCfit_withGP()

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib

# Bands to plot: None = all SN.avail_filters; or pass an ordered list
FILTERS = None

# Legend strings per internal band key
LEGEND_MAP = {}

def LEGEND_LABEL_FUNC(band_key: str) -> str:
    if "swift" in band_key:
        return band_key.split("_")[1] + "(Swift)"
    if "_" in band_key:
        return band_key.split("_")[1]
    return str(band_key)

def legend_label_for_band(band_key: str) -> str:
    return LEGEND_MAP.get(band_key, LEGEND_LABEL_FUNC(band_key))

# Exactly 16:9 aspect ratio for presentations
FIGSIZE = (16, 9) 
DPI = 500

# ==========================================
# WATERFALL & SPACE CONTROLS
# ==========================================
# Offset for magnitudes (e.g., 1.5 or 2.0 works well for staggering bands)
OFFSET_PER_BAND = 1.5     

PLOT_Y_MAGNITUDE = True      # True = plot Magnitude (Proper AB mag), False = plot Log Flux
INVERT_Y_AXIS = True         # True = Inverts the Y axis (standard for magnitudes so brighter is up)
PLOT_X_LINEAR = True         # True = plot linear Phase (days), False = plot Log Phase
ANNOTATE_BANDS = False       # True = labels the band name at the end of its GP curve

# DYNAMIC SORTING CONTROLS
SORT_BLUE_TO_RED = True      # True = sorts bands from shortest to longest wavelength

# TRUNCATION
TRUNCATE_GP_TO_DATA = True   # True = slices GP to only span between first and last real data points

# HOW SKINNY SHOULD THE PLOTS BE?
# 2.5 means "make the box 2.5 times taller than it is wide". Increase this to squeeze further.
PLOT_ASPECT_RATIO = 2.5 
# ==========================================

# Set manual limits here, or set to None for automatic data-driven limits
XLIM = (0, 10)    
YLIM = (15, 55)

SHOW_SPECTRA_LINES = False
SPECTRA_KW = dict(linestyle="-", color="k", lw=0.8, alpha=0.3, label="Spectra")

GP_FILL_ALPHA = 0.1
MARKER_MS = 5

LEGEND_KW = dict(
    fontsize=14,
    ncol=1, 
    loc="upper left",
    fancybox=True,
    framealpha=0.5,
)
LEGEND_TITLE = None  

TITLE = None  

SAVE_FIG = None # e.g. '/path/to/save.png'

# --- end knobs ---

if not hasattr(SN, "fitted_phot"):
    raise RuntimeError("Define SN.fitted_phot first (run SN.LCfit_withGP()).")

# 1. Gather all available bands
initial_bands = np.array(SN.avail_filters, dtype=object).tolist() if FILTERS is None else list(FILTERS)
valid_bands = [b for b in initial_bands if b in SN.fitted_phot]

# 2. Approximate central wavelengths in Angstroms
WAVE_DICT = {
    'w2': 1928, 'm2': 2246, 'w1': 2600, 
    'u': 3543, 'b': 4450, 'g': 4770, 'v': 5510, 
    'r': 6231, 'i': 7625, 'z': 9134, 'y': 10040, 
    'j': 12200, 'h': 16300, 'k': 21900, 'ks': 21900
}

def get_wave(band_name):
    b_lower = band_name.lower().split('_')[-1]
    if 'swift' in band_name.lower():
        b_lower = band_name.lower().split('_')[1]
        
    for key, wave in sorted(WAVE_DICT.items(), key=lambda item: len(item[0]), reverse=True):
        if key in b_lower:
            return wave
            
    print(f"Warning: No wavelength found for {band_name}. Defaulting to 6000A for math.")
    return 6000.0

# 3. Sort bands from bluest to reddest
if SORT_BLUE_TO_RED:
    plot_bands = sorted(valid_bands, key=get_wave)
else:
    plot_bands = valid_bands

# 4. Split bands into 3 equal (or near-equal) chunks for the 3 panels
band_chunks = np.array_split(plot_bands, 3)

plt.rc("font", family="serif")
plt.rc("xtick", labelsize=13)
plt.rc("ytick", labelsize=13)

# 5. Initialize 3-panel figure with shared y-axis
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, dpi=DPI, sharey=True)
norm = plt.Normalize(vmin=1500, vmax=22000)

for ax_idx, (ax, chunk) in enumerate(zip(axes, band_chunks)):
    
    # Force the physical aspect ratio of the subplot box to squeeze it
    ax.set_box_aspect(PLOT_ASPECT_RATIO)
    
    for i, f in enumerate(chunk):
        
        # Force color mapping entirely by wavelength (bluest to reddest)
        wave = get_wave(f)
        c = cm.Spectral_r(norm(wave))

        log_phase, log_flux, err_log_flux, sudo_mask = SN.fitted_phot[f]["clipped_extended_data"]
        new_log_phase, mu, std = SN.fitted_phot[f]["fit_highcadence"]

        log_phase = np.asarray(log_phase, dtype=float)
        log_flux = np.asarray(log_flux, dtype=float)
        err_log_flux = np.asarray(err_log_flux, dtype=float)
        sudo_mask = np.asarray(sudo_mask, dtype=bool)
        
        good = ~sudo_mask

        lab = legend_label_for_band(f)
        try:
            mk = mark_dict[f]
        except (NameError, KeyError):
            mk = "o"

        # Local offset (resets for each panel so they stack identically)
        current_offset = i * OFFSET_PER_BAND

        # ---------------------------------------------------------
        # Space Transformations + Proper AB Magnitude Conversion
        # ---------------------------------------------------------
        if PLOT_X_LINEAR:
            x_plot = 10**log_phase
            x_gp_plot = 10**new_log_phase
        else:
            x_plot = log_phase
            x_gp_plot = new_log_phase

        if PLOT_Y_MAGNITUDE:
            eff_wave = get_wave(f)
            zp_correction = -5.0 * np.log10(eff_wave) - 2.408
            
            y_plot = -2.5 * log_flux + zp_correction + current_offset
            y_err_plot = 2.5 * err_log_flux
            y_err_good = y_err_plot[good]
            
            y_gp_mu_plot = -2.5 * mu + zp_correction + current_offset
            y_gp_lower = -2.5 * (mu + std) + zp_correction + current_offset
            y_gp_upper = -2.5 * (mu - std) + zp_correction + current_offset
            
        else:
            y_plot = log_flux + current_offset
            y_err_plot = err_log_flux
            y_err_good = y_err_plot[good]
            
            y_gp_mu_plot = mu + current_offset
            y_gp_upper = mu + std + current_offset
            y_gp_lower = mu - std + current_offset
        # ---------------------------------------------------------

        if TRUNCATE_GP_TO_DATA and np.any(good):
            min_x = np.min(x_plot[good])
            max_x = np.max(x_plot[good])
            
            valid_gp_mask = (x_gp_plot >= min_x) & (x_gp_plot <= max_x)
            
            x_gp_plot = x_gp_plot[valid_gp_mask]
            y_gp_mu_plot = y_gp_mu_plot[valid_gp_mask]
            y_gp_upper = y_gp_upper[valid_gp_mask]
            y_gp_lower = y_gp_lower[valid_gp_mask]

        # Plot actual photometry
        if np.any(good):
            _eb = ax.errorbar(
                x_plot[good], y_plot[good], yerr=y_err_good,
                fmt=mk, mfc=c, ms=MARKER_MS, color=c,
                linestyle="None", label="_nolegend_"
            )
            data_color = _eb[0].get_color()
        else:
            data_color = c

        # Plot GP fit
        if len(x_gp_plot) > 0:
            ax.plot(x_gp_plot, y_gp_mu_plot, color=data_color)
            ax.fill_between(x_gp_plot, y_gp_upper, y_gp_lower, color=data_color, alpha=GP_FILL_ALPHA)

        # Annotate band names
        if ANNOTATE_BANDS and len(x_gp_plot) > 0:
            ax.text(x_gp_plot[-1] + (0.05 if not PLOT_X_LINEAR else 0.15), y_gp_mu_plot[-1], 
                    f"{lab}", color=data_color, fontsize=12, va='center', ha='left')

    if XLIM is not None:
        ax.set_xlim(XLIM)
        
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis='both', which='minor', labelsize=14)
    ax.grid(True)
    
    if SHOW_SPECTRA_LINES:
        spec_log_phases = SN.get_spec_log_phase()
        if len(spec_log_phases) > 0:
            y0, y1 = ax.get_ylim()
            x_lines = 10**spec_log_phases if PLOT_X_LINEAR else spec_log_phases
            ax.vlines(x_lines, y0, y1, **SPECTRA_KW)

# Handle Global Y-Axis Limits and Inversion on the first axis (it will propogate to all due to sharey=True)
if YLIM is not None:
    axes[0].set_ylim(YLIM)

if INVERT_Y_AXIS and PLOT_Y_MAGNITUDE:
    axes[0].invert_yaxis()

# Set Labels (X on middle plot, Y on left plot)
x_label = r"Phase relative to explosion (days)" if PLOT_X_LINEAR else r"Log Phase relative to explosion (days)"
axes[1].set_xlabel(x_label, fontsize=18)

if PLOT_Y_MAGNITUDE:
    y_label = r"Apparent AB Magnitude + Offset" if OFFSET_PER_BAND != 0 else r"Apparent AB Magnitude"
else:
    y_label = r"Log Flux + Offset" if OFFSET_PER_BAND != 0 else r"Log Flux ($\log_{10}(\rm{erg \ s^{-1} cm^{-2} \AA^{-1}})$)"
axes[0].set_ylabel(y_label, fontsize=18)

if TITLE is not None:
    fig.suptitle(TITLE, fontsize=18)

# --- Custom Legend Setup ---
custom_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
           markersize=MARKER_MS+2, label='Actual Photometry'),
    Line2D([0], [0], color='black', linestyle='-', lw=2, label='GP Fit')
]

leg = axes[0].legend(handles=custom_handles, **LEGEND_KW)
if LEGEND_TITLE is not None:
    leg.set_title(LEGEND_TITLE)

# --- NEW: Explicit Layout and Tethered Colorbar Creation ---

sm = plt.cm.ScalarMappable(cmap=cm.Spectral_r, norm=norm)
sm.set_array([])

# 1. Apply tight_layout. Because of ax.set_box_aspect(), Matplotlib will naturally 
# compress the plots horizontally into the center of the 16:9 canvas.
fig.tight_layout(w_pad=0.0)

# 2. Get the physical coordinates of the right-most subplot AFTER it has been squeezed
pos = axes[-1].get_position()

# 3. Create a dedicated axis for the colorbar tethered to the skinnier plots
# pos.x1 is the exact right edge of the right-most plot. We add 0.01 for a tiny gap.
cbar_ax = fig.add_axes([pos.x1 + 0.01, pos.y0, 0.015, pos.height])

# 4. Draw the colorbar into that dedicated axis
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(r'Effective Wavelength ($\mathrm{\AA}$)', fontsize=18)
cbar.ax.tick_params(labelsize=14)

if SAVE_FIG is not None:
    fig.savefig(SAVE_FIG, dpi=DPI, bbox_inches="tight")
    print("Saved:", SAVE_FIG)

plt.show()